In [0]:
from pyspark.sql.functions import col,unix_timestamp,round,year,month,dayofmonth,hour,dayofweek,when
from delta.tables import DeltaTable

In [0]:
bronze_table = spark.table("nyc_taxi.lakehouse.bronze_taxi")

In [0]:
silver_df = bronze_table.dropna(subset= [ "pickup_datetime",
        "dropoff_datetime",
        "trip_distance",
        "fare_amount"
    ])

In [0]:
silver_df = silver_df.filter(col("fare_amount") > 0).filter(col("trip_distance") > 0).filter(col("pickup_datetime") < col("dropoff_datetime")).filter(col("passenger_count") > 0 )

In [0]:
silver_df = silver_df.withColumn("trip_duration_minutes",
                                 round((
                                     unix_timestamp(col("dropoff_datetime"))
                                     -
                                     unix_timestamp(col("pickup_datetime"))

                                 )/60))

In [0]:
silver_df = (
    silver_df.withColumn("pickup_year",year(col("pickup_datetime")))
    .withColumn("pickup_month",month(col("pickup_datetime")))
    .withColumn("pickup_day",dayofmonth(col("pickup_datetime")))
    .withColumn("pickup_hour",hour(col("pickup_datetime")))
    .withColumn("pickup_day_of_week",dayofweek(col("pickup_datetime")))
)

In [0]:
silver_df = silver_df.withColumn("fare_per_mile", round(
    col("fare_amount")/col("trip_distance"),2
))

In [0]:
silver_df = silver_df.withColumn("trip_catogery",
                                 when(col("trip_distance") < 2, "short")
                                 .when(col("trip_distance") < 10, "medium")
                                 .otherwise("long"))

In [0]:
# silver_df.write.format("delta").mode("overwrite")\
#     .option("overwriteSchema", "true") \
#     .partitionBy("pickup_year","pickup_month")\
#     .saveAsTable("nyc_taxi.lakehouse.silver_taxi")

In [0]:
silver_table = DeltaTable.forName(
    spark,
    "nyc_taxi.lakehouse.silver_taxi"
)
(
    silver_table.alias("target")
    .merge(
        silver_df.alias("source"),
        "target.trip_id = source.trip_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)